# Combined Model Full-Track Waterfall

Loads `LitCombinedDataModule` and `LitS4CombinedModel` from the training
checkpoint, reads the **full-length** I/Q track for a test-set event directly
from the HDF5 file (bypassing the `cutoff` truncation used during training),
slides the denoiser across the track in non-overlapping windows of `cutoff`
samples, and produces a time vs. frequency waterfall plot.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import h5py
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, fftshift

from src.models.model import LitS4CombinedModel
from src.models.networks import S4DCombinedModel
from src.data.data import LitCombinedDataModule
from src.utils.noise import noise_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Configuration

In [ ]:
# ── Update this path ──────────────────────────────────────────────────────────
path            = '../runs/combined_denoise_regression/lightning_logs/<run_id>'
CHECKPOINT_PATH = os.path.join(path, 'checkpoints/last.ckpt')
CONFIG_PATH     = os.path.join(path, '../../config.yaml')
# ─────────────────────────────────────────────────────────────────────────────

# Index into the test set (0 = first test track)
TEST_TRACK_IDX = 0

# Maximum number of segments to process (None = use the full track length)
MAX_SEGMENTS = None

# Baseband half-width to display around the carrier frequency [Hz]
DISPLAY_BW_HZ = 6e6   # ±6 MHz → 12 MHz total display bandwidth

# Sampling rate (must match noise_model in src/utils/noise.py)
FS = 403e6   # Hz

## 2. Load Data Module from Checkpoint

In [ ]:
dm = LitCombinedDataModule.load_from_checkpoint(CHECKPOINT_PATH)
test_loader = dm.test_dataloader()

CUTOFF       = dm.hparams.cutoff
NOISE_CONST  = dm.hparams.noise_const
APPLY_FILTER = dm.hparams.get('apply_filter', False)
carrier_idx  = dm.observables.index('avg_carrier_frequency_Hz')

print(f'Test batches  : {len(test_loader)}')
print(f'Input channels: {dm.inputs}')
print(f'Targets       : {dm.variables}')
print(f'Observables   : {dm.observables}')
print(f'Cutoff        : {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs per segment)')
print(f'Noise const   : {NOISE_CONST}')

## 3. Load Model from Checkpoint

In [ ]:
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

enc_cfg = config['model']['init_args']['encoder']['init_args']

encoder = S4DCombinedModel(
    d_input            = enc_cfg.get('d_input', 2),
    d_output           = enc_cfg.get('d_output', 2),
    denoiser_d_model   = enc_cfg.get('denoiser_d_model', 128),
    denoiser_n_layers  = enc_cfg.get('denoiser_n_layers', 6),
    denoiser_dropout   = enc_cfg.get('denoiser_dropout', 0.0),
    denoiser_prenorm   = enc_cfg.get('denoiser_prenorm', False),
    regressor_d_model  = enc_cfg.get('regressor_d_model', 128),
    regressor_n_layers = enc_cfg.get('regressor_n_layers', 6),
    regressor_dropout  = enc_cfg.get('regressor_dropout', 0.0),
    regressor_prenorm  = enc_cfg.get('regressor_prenorm', False),
    regressor_fc_hidden= enc_cfg.get('regressor_fc_hidden', [64, 32]),
)

model = LitS4CombinedModel.load_from_checkpoint(CHECKPOINT_PATH, encoder=encoder)
model = model.to(device).eval()
print('Model loaded successfully')
print(f'  Denoiser : d_model={enc_cfg.get("denoiser_d_model", 128)}, '
      f'n_layers={enc_cfg.get("denoiser_n_layers", 6)}')
print(f'  Regressor: d_model={enc_cfg.get("regressor_d_model", 128)}, '
      f'n_layers={enc_cfg.get("regressor_n_layers", 6)}')

## 4. Identify and Read the Full Test Track

The DataModule gives us the test-set indices and the path to each HDF5 row.
We read the **complete** row — not truncated to `cutoff` — so the waterfall
covers the entire track length.

In [ ]:
# Resolve the test-track index via the DataModule's internal index
global_idx = dm.test_dataset.active_indices[TEST_TRACK_IDX]
hdf5_path, local_row = dm.test_dataset._index[global_idx]
print(f'Test track {TEST_TRACK_IDX}  →  {os.path.basename(hdf5_path)}, row {local_row}')

# Read the FULL-LENGTH track (no cutoff truncation)
with h5py.File(hdf5_path, 'r') as f:
    track_I = f[dm.inputs[0]][local_row][:].astype(np.float32)
    track_Q = f[dm.inputs[1]][local_row][:].astype(np.float32)
    carrier_freq_hz = float(f['avg_carrier_frequency_Hz'][local_row])

track_len  = len(track_I)
n_segments = track_len // CUTOFF
if MAX_SEGMENTS is not None:
    n_segments = min(n_segments, MAX_SEGMENTS)

print(f'Full track length : {track_len:,} samples  ({track_len / FS * 1e3:.3f} ms)')
print(f'Segments          : {n_segments}  ×  {CUTOFF} samples  ({CUTOFF / FS * 1e6:.1f} µs each)')
print(f'Carrier freq      : {carrier_freq_hz / 1e9:.6f} GHz')

## 5. Apply Combined Model Segment by Segment

Each `cutoff`-length window is noised and normalised identically to
`Project8SimCombined.__getitem__`, then passed through the model.
`forward()` returns `(x_denoised, y_pred)`; we keep the denoised I/Q.

In [ ]:

# Add noise to the full track first, then divide into segments
noisy_I_full = track_I + noise_model(len(track_I), NOISE_CONST)
noisy_Q_full = track_Q + noise_model(len(track_Q), NOISE_CONST)
if APPLY_FILTER:
    from src.utils.noise import bandpass_filter
    noisy_I_full = bandpass_filter(noisy_I_full)
    noisy_Q_full = bandpass_filter(noisy_Q_full)

denoised_I = np.zeros(n_segments * CUTOFF, dtype=np.float32)
denoised_Q = np.zeros(n_segments * CUTOFF, dtype=np.float32)
noisy_I    = np.zeros(n_segments * CUTOFF, dtype=np.float32)
noisy_Q    = np.zeros(n_segments * CUTOFF, dtype=np.float32)

for s in range(n_segments):
    start = s * CUTOFF
    end   = start + CUTOFF

    Xn_I = noisy_I_full[start:end]
    Xn_Q = noisy_Q_full[start:end]

    # Normalise per channel by std of the noisy segment (mirrors __getitem__)
    std_I = float(np.std(Xn_I)) + 1e-8
    std_Q = float(np.std(Xn_Q)) + 1e-8

    X_norm = np.stack([Xn_I / std_I, Xn_Q / std_Q], axis=-1)  # (cutoff, 2)

    noisy_I[start:end] = X_norm[:, 0]
    noisy_Q[start:end] = X_norm[:, 1]

    # Run model — forward returns (x_denoised, y_pred)
    x_in = torch.tensor(X_norm, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        x_den_norm, _y_pred = model(x_in)
    x_den_norm = x_den_norm.squeeze(0).cpu().numpy()  # (cutoff, 2)

    # Denormalise back to original amplitude scale
    denoised_I[start:end] = x_den_norm[:, 0] * std_I
    denoised_Q[start:end] = x_den_norm[:, 1] * std_Q

    if (s + 1) % max(1, n_segments // 10) == 0:
        print(f'  {s+1:4d} / {n_segments} segments processed')

print('Done.')


## 6. Compute Power Spectra

FFT of the complex signal `I + jQ` for each window; normalise each column
by its median power to get a per-segment noise-floor estimate.

In [ ]:
# Frequency axis centred on the carrier
baseband_freqs_hz  = fftshift(fftfreq(CUTOFF, d=1.0 / FS))
physical_freqs_ghz = (carrier_freq_hz + baseband_freqs_hz) / 1e9
freq_mask          = np.abs(baseband_freqs_hz) <= DISPLAY_BW_HZ
n_display          = int(freq_mask.sum())

# Time axis: centre of each segment
seg_dt_s    = CUTOFF / FS
seg_times_s = np.arange(n_segments) * seg_dt_s + seg_dt_s / 2

def build_power_map(I_arr, Q_arr):
    """Return (n_display, n_segments) power array."""
    pmap = np.zeros((n_display, n_segments), dtype=np.float32)
    for s in range(n_segments):
        start = s * CUTOFF
        cplx  = I_arr[start:start + CUTOFF] + 1j * Q_arr[start:start + CUTOFF]
        spec  = fftshift(np.abs(fft(cplx)) ** 2)
        pmap[:, s] = spec[freq_mask]
    return pmap

def scale_snr(power_map):
    noise_floor = np.median(power_map, axis=0, keepdims=True)
    noise_floor = np.where(noise_floor > 0, noise_floor, 1.0)
    snr = power_map / noise_floor
    return np.clip(snr / np.percentile(snr, 99.9), 0.0, 1.0)

snr_den = scale_snr(build_power_map(denoised_I, denoised_Q))
snr_noi = scale_snr(build_power_map(noisy_I,    noisy_Q))

# Mesh edges for pcolormesh
t_edges = np.append(seg_times_s - seg_dt_s / 2,
                    seg_times_s[-1] + seg_dt_s / 2)
f_ghz   = physical_freqs_ghz[freq_mask]
df_ghz  = np.abs(f_ghz[1] - f_ghz[0])
f_edges = np.append(f_ghz - df_ghz / 2, f_ghz[-1] + df_ghz / 2)

print(f'Power map shape   : {snr_den.shape}')
print(f'Frequency range   : {f_ghz.min():.4f} – {f_ghz.max():.4f} GHz')
print(f'Time range        : {t_edges[0]:.6f} – {t_edges[-1]:.6f} s')

## 7. Waterfall Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

mesh = ax.pcolormesh(
    t_edges, f_edges, snr_den,
    cmap='Blues', vmin=0.0, vmax=1.0,
    shading='flat', rasterized=True,
)
cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=11)
cbar.set_ticks(np.linspace(0.1, 1.0, 10))

ax.set_xlabel('Time [s]', fontsize=12)
ax.set_ylabel('Frequency [GHz]', fontsize=12)
ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX}', fontsize=13)

fig.tight_layout()
plt.show()

## 8. Side-by-side Comparison: Noisy vs. Denoised

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, data, title in [
    (axes[0], snr_noi, 'Noisy input'),
    (axes[1], snr_den, 'Denoised (combined model)'),
]:
    mesh = ax.pcolormesh(
        t_edges, f_edges, data,
        cmap='Blues', vmin=0.0, vmax=1.0,
        shading='flat', rasterized=True,
    )
    cbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    cbar.set_label('Scaled signal to noise ratio (linear)', fontsize=10)
    cbar.set_ticks(np.linspace(0.1, 1.0, 10))
    ax.set_xlabel('Time [s]', fontsize=12)
    ax.set_ylabel('Frequency [GHz]', fontsize=12)
    ax.set_title(f'Project 8 – Event {TEST_TRACK_IDX} ({title})', fontsize=12)

fig.tight_layout()
plt.show()